# Project Milestone Two

**Data Preparation and Model Exploration**

**Note: No late assignments accepted, we need the time to grade them!**

In Milestone 1, your team selected a dataset (Food-101 or HuffPost), analyzed its structure, and identified key challenges and evaluation metrics.
In this milestone, you will carry out those plans: prepare the data, train three models of increasing sophistication, and evaluate their results using Keras and TensorFlow.
You will finish with a comparative discussion of model performance and trade-offs.


### Submission Guidelines

* Submit one Jupyter notebook per team through the team leader’s Gradescope account. **Include all team members names at the top of the notebook.**
* Include all code, plots, and answers inline below.
* Ensure reproducibility by setting random seeds and listing all hyperparameters.
* Document any AI tools used, as required by the CDS policy.


## Problem 1 – Data Preparation and Splits (20 pts)

### Goals

Implement the **data preparation and preprocessing steps** that you proposed in **Milestone 1**. You’ll clean, normalize, and split your data so that it’s ready for modeling and reproducible fine-tuning.

### Steps to Follow

1. **Load your chosen dataset**

   * Use `datasets.load_dataset()` from **Hugging Face** to load **Food-101** or **HuffPost**.
   * Display basic information (e.g., number of samples, feature names, example entries).

2. **Apply cleaning and normalization**

   * **Images:**

     * Ensure all images are in RGB format.
     * Resize or crop to a consistent shape (e.g., `224 × 224`).
     * Drop or fix any corrupted files.
   * **Text:**

     * Concatenate headline + summary (for HuffPost).
     * Strip whitespace, convert to lowercase if appropriate, and remove empty samples.
     * Optionally remove duplicates or extremely short entries.

3. **Standardize or tokenize the inputs**

   * **Images:**

     * Normalize pixel values (e.g., divide by 255.0).
     * Define a minimal augmentation pipeline (e.g., random flip, crop, or rotation).
   * **Text:**

     * Create a tokenizer or `TextVectorization` layer.
     * Set a target `max_length` based on your analysis from Milestone 1 (e.g., 95th percentile).
     * Apply padding/truncation and build tensors for input + labels.

4. **Handle dataset-specific challenges**

   * If you identified **class imbalance**, compute label counts and, if needed, create a dictionary of `class_weights`.
   * If you noted **length or size variance**, verify that your truncation or resizing works as intended.
   * If you planned **noise filtering**, include the cleaning step and briefly explain your criteria (e.g., remove items with missing text or unreadable images).

5. **Create reproducible splits**

   * Split your cleaned dataset into **train**, **validation**, and **test** subsets (e.g., 80 / 10 / 10).
   * Use a fixed random seed for reproducibility (`random_seed = 42`).
   * Use **stratified splits**  (e.g., with `train_test_split` and `stratify = labels`).
   * Display the size of each subset.

6. **Document your pipeline**

   * Summarize your preprocessing steps clearly in Markdown or code comments.
   * Save or display a few representative examples after preprocessing to confirm the transformations are correct.




In [ ]:
import os, time, json, gc, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import load_dataset

import tensorflow as tf
from tensorflow import keras

# — Sergey: Metal GPU adds dispatch overhead on small models (Embedding+Dense),
#   making them ~10x slower. Set False for P2-P3, True for P4 (BERT).
USE_GPU = True

if USE_GPU:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU enabled: {len(gpus)} device(s)")
    else:
        print("No GPU found — CPU only")
else:
    tf.config.set_visible_devices([], 'GPU')
    print("GPU disabled (CPU mode)")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


Special provision for colab!

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install "transformers<5" tf-keras tensorflow "huggingface_hub<0.28"
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU
    COLAB_CPU = True
else:
    COLAB_CPU = False

### Load & Inspect

In [ ]:
URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
huff_all = load_dataset("json", data_files=URL, split="train")

print(huff_all)
print(f"Columns: {huff_all.column_names}")
print(f"Total samples: {len(huff_all):,}")

df = huff_all.to_pandas()
df.head(3)

### Clean & Merge Categories

In [ ]:
# Merge confusable categories identified in Milestone 1
LABEL_MAP = {
    "ARTS & CULTURE": "ARTS", "CULTURE & ARTS": "ARTS",
    "THE WORLDPOST": "WORLD NEWS", "WORLDPOST": "WORLD NEWS",
    "PARENTS": "PARENTING",
    "STYLE & BEAUTY": "STYLE",
    "TASTE": "FOOD & DRINK",
    "GREEN": "ENVIRONMENT",
    "COLLEGE": "EDUCATION",
    "HEALTHY LIVING": "HOME & LIVING",
}

print(f"Before merge: {df['category'].nunique()} categories, {len(df):,} rows")

df["category"] = df["category"].replace(LABEL_MAP)

# Combine headline + description
df["text"] = df.apply(
    lambda r: (r["headline"] + " [SEP] " + r["short_description"]).strip()
    if pd.notna(r["short_description"]) and r["short_description"].strip()
    else r["headline"],
    axis=1,
)

# Drop empty headlines
empty_mask = df["headline"].isna() | (df["headline"].str.strip() == "")
print(f"Empty headlines dropped: {empty_mask.sum()}")
df = df[~empty_mask].copy()

# Remove exact duplicates (headline + short_description)
before_dedup = len(df)
df = df.drop_duplicates(subset=["headline", "short_description"], keep="first")
print(f"Duplicates removed: {before_dedup - len(df)}")

print(f"After cleaning: {df['category'].nunique()} categories, {len(df):,} rows")

### Class Distribution

In [ ]:
counts = df["category"].value_counts()

fig, ax = plt.subplots(figsize=(10, 8))
counts.plot.barh(ax=ax)
ax.set_xlabel("Count")
ax.set_title("Class Distribution After Merging")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"Categories: {len(counts)}")
print(f"Largest:  {counts.index[0]} ({counts.iloc[0]:,})")
print(f"Smallest: {counts.index[-1]} ({counts.iloc[-1]:,})")
print(f"Median:   {int(counts.median()):,}")
print(f"Imbalance ratio (max/min): {counts.iloc[0] / counts.iloc[-1]:.1f}x")

### Stratified Train / Val / Test Splits

In [ ]:
texts = np.array(df["text"].tolist(), dtype=object)
labels = df["category"].tolist()

# Encode labels
class_names = sorted(set(labels))
label2id = {name: i for i, name in enumerate(class_names)}
y = np.array([label2id[c] for c in labels])
num_classes = len(class_names)

# 80/10/10 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, y, test_size=0.2, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")
print(f"Classes: {num_classes}")

# Compute class weights for imbalanced training
cw = compute_class_weight("balanced", classes=np.arange(num_classes), y=y_train)
class_weights = dict(enumerate(cw))
print(f"Class weight range: [{min(cw):.3f}, {max(cw):.3f}]")

### Text Length Distribution

In [ ]:
word_counts = pd.Series([len(t.split()) for t in X_train])

fig, ax = plt.subplots(figsize=(8, 4))
word_counts.hist(bins=60, ax=ax, edgecolor="black", alpha=0.7)
ax.axvline(word_counts.quantile(0.95), color="red", ls="--", label=f"P95 = {word_counts.quantile(0.95):.0f}")
ax.axvline(word_counts.median(), color="orange", ls="--", label=f"Median = {word_counts.median():.0f}")
ax.set_xlabel("Word count")
ax.set_ylabel("Frequency")
ax.set_title("Text Length Distribution (train set)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean:   {word_counts.mean():.1f}")
print(f"Median: {word_counts.median():.0f}")
print(f"P95:    {word_counts.quantile(0.95):.0f}")
print(f"P99:    {word_counts.quantile(0.99):.0f}")
print(f"Max:    {word_counts.max()}")

### Sample Examples

In [ ]:
for i in range(5):
    print(f"[{class_names[y_train[i]]}] {X_train[i][:120]}...")
    print()

### Graded Questions (5 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible.

1. **Data Loading and Cleaning:**
   Describe how you loaded your dataset and the key cleaning steps you implemented (e.g., handling missing data, normalizing formats, or removing duplicates).



1.1. We loaded the HuffPost News Category Dataset (200,853 records, 41 categories) from a JSON mirror on HuggingFace using `load_dataset("json", ...)`. Three cleaning steps were applied: (1) merged 8 groups of overlapping categories identified in Milestone 1 (e.g., ARTS & CULTURE / CULTURE & ARTS → ARTS, PARENTS → PARENTING), reducing 41 categories to 31; (2) dropped 6 rows with empty headlines; (3) removed 484 exact duplicate (headline + short_description) pairs. Final cleaned dataset: 200,363 rows across 31 categories.

2. **Preprocessing and Standardization:**
   Summarize your preprocessing pipeline. Include any normalization, tokenization, resizing, or augmentation steps, and explain why each was necessary for your dataset.
  

1.2. Each sample's text was formed by concatenating the headline and short_description with a `[SEP]` separator; when the description was missing (~9.8% of records), the headline alone was used. A `TextVectorization` layer was configured with `max_tokens=20,000` and `output_sequence_length=64`, chosen based on the computed P95 word count of 57 words in the training set (mean 30.2, median 29). The layer was adapted on training text only, then applied once to produce integer-encoded numpy arrays (shape 160,290 × 64), avoiding repeated tokenization during training. No text augmentation was applied at this stage.

3. **Train/Validation/Test Splits:**
   Explain how you divided your data into subsets, including the split ratios, random seed, and any stratification or leakage checks you used to verify correctness.


1.3. The cleaned dataset was split into train (80%), validation (10%), and test (10%) subsets using two successive calls to `train_test_split` with `random_state=42` and `stratify=y` to preserve class proportions. This produced 160,290 training, 20,036 validation, and 20,037 test samples. Deduplication was performed before splitting to prevent data leakage — any exact (headline + description) duplicates that might have landed in both train and test were removed beforehand. The stratified split ensures that even the smallest class (LATINO VOICES, 1,129 total) is proportionally represented in all three subsets.

4. **Class Distribution and Balance:**
   Report your label counts and describe any class imbalances you observed. If applicable, explain how you addressed them (e.g., weighting, oversampling, or data augmentation).


1.4. After merging, the largest class is POLITICS (32,721 samples) and the smallest is LATINO VOICES (1,129), giving an imbalance ratio of 29.0x. The median class size is 3,876, well below the mean (~6,463), confirming a long-tailed distribution visible in the bar chart. To address this, we computed per-class weights using `sklearn`'s `compute_class_weight("balanced")`, which produced weights ranging from 0.198 (for POLITICS) to 5.726 (for LATINO VOICES). These weights are passed to `model.fit()` via the `class_weight` parameter so the loss function penalizes errors on minority classes proportionally more.

## Problem 2 – Baseline Model (20 pts)

### Goal

Build and train a **simple, fully functional baseline model** to establish a reference level of performance for your dataset.
This baseline will help you evaluate whether later architectures and fine-tuning steps actually improve results.


### Steps to Follow

1. **Construct a baseline model**

   * **Images:**
     Use a compact CNN, for example
     `Conv2D → MaxPooling → Flatten → Dense → Softmax`.
   * **Text:**
     Use a small embedding-based classifier such as
     `Embedding → GlobalAveragePooling → Dense → Softmax`.
   * Keep the model small enough to train in minutes on Colab.

2. **Compile the model**

   * Optimizer: `Adam` or `AdamW`.
   * Loss: `categorical_crossentropy` (for multi-class).
   * Metrics: at least `accuracy`; add `F1` if appropriate.

3. **Train and validate**

   * Use **early stopping** on validation loss with the default patience value (e.g., 5 epochs).
   * Record number of epochs trained and total runtime.

4. **Visualize results**

   * Plot **training vs. validation accuracy and loss**.
   * Carefully observe: does the model underfit, overfit, or generalize reasonably?

5. **Report baseline performance**

   * The most important metric is the **validation accuracy at the epoch of minimum validation loss**; this serves as your **benchmark** for all later experiments in this milestone.
   * Evaluate on the **test set** and record final metrics.

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

RESULTS_FILE = "results/ms2_results.json"
results = {}

def save_results():
    os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Results saved ({len(results)} models)")

def load_results():
    global results
    try:
        with open(RESULTS_FILE) as f:
            results = json.load(f)
        if results:
            print(f"Restored {len(results)} results from {RESULTS_FILE}:")
            for name, r in results.items():
                print(f"  {name}: test_acc={r['test_acc']:.4f}, test_f1={r['test_f1']:.4f}")
    except FileNotFoundError:
        pass

def evaluate_model(model, name, X_te, y_te, hist, train_time, batch_size=512):
    y_pred = model.predict(X_te, batch_size=batch_size, verbose=0).argmax(axis=-1)
    test_acc = float(np.mean(y_pred == y_te))
    test_f1 = float(f1_score(y_te, y_pred, average="macro"))

    val_losses = hist.history["val_loss"]
    best_epoch = int(np.argmin(val_losses)) + 1
    val_acc = float(hist.history["val_accuracy"][best_epoch - 1])
    val_loss = float(val_losses[best_epoch - 1])

    results[name] = {
        "val_acc": val_acc, "val_loss": val_loss, "best_epoch": best_epoch,
        "test_acc": test_acc, "test_f1": test_f1,
        "params": model.count_params(), "train_time": train_time,
    }
    save_results()

    print(f"=== {name} ===")
    print(f"Best epoch:  {best_epoch}")
    print(f"Val acc:     {val_acc:.4f}  |  Val loss: {val_loss:.4f}")
    print(f"Test acc:    {test_acc:.4f}  |  Test F1:  {test_f1:.4f}")
    print(f"Params:      {model.count_params():,}")
    print(f"Train time:  {format_hms(train_time)}")
    return y_pred

# Auto-restore on cell run
load_results()

### Tokenization

In [ ]:
MAX_TOKENS = 20_000
SEQ_LEN = 64  # based on P95 word count from the distribution above
BATCH = 256

text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=SEQ_LEN,
    standardize="lower_and_strip_punctuation",
)
text_vectorizer.adapt(X_train)

vocab_size = text_vectorizer.vocabulary_size()
print(f"Vocabulary size: {vocab_size:,}")
print(f"Sequence length: {SEQ_LEN}")

# Pre-vectorize once into numpy — avoids re-tokenizing every epoch
X_train_vec = text_vectorizer(X_train).numpy()
X_val_vec   = text_vectorizer(X_val).numpy()
X_test_vec  = text_vectorizer(X_test).numpy()

print(f"Vectorized shapes — train: {X_train_vec.shape}, val: {X_val_vec.shape}, test: {X_test_vec.shape}")

### Build & Compile Baseline

In [ ]:
baseline = tf.keras.Sequential([
    tf.keras.layers.Embedding(MAX_TOKENS, 64),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
], name="baseline_emb_avg")

baseline.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

baseline.summary()

### Train Baseline

In [ ]:
es = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

t0 = time.time()
hist_baseline = baseline.fit(
    X_train_vec, y_train,
    validation_data=(X_val_vec, y_val),
    epochs=50,
    batch_size=BATCH,
    class_weight=class_weights,
    callbacks=[es],
    verbose=1,
)
train_time = time.time() - t0
print(f"Training time: {format_hms(train_time)}")

### Learning Curves & Evaluation

In [ ]:
def plot_learning_curves(hist, title):
    val_losses = hist.history["val_loss"]
    min_val_loss = min(val_losses)
    best_epoch = val_losses.index(min_val_loss)
    val_acc_best = hist.history["val_accuracy"][best_epoch]

    epochs = range(1, len(val_losses) + 1)
    fig, axs = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

    axs[0].plot(epochs, hist.history["loss"], label="train")
    axs[0].plot(epochs, val_losses, label="val")
    axs[0].scatter(best_epoch + 1, min_val_loss, color="red", marker="x", s=60, label="best")
    axs[0].set_title(f"{title} — Loss")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True)

    axs[1].plot(epochs, hist.history["accuracy"], label="train")
    axs[1].plot(epochs, hist.history["val_accuracy"], label="val")
    axs[1].scatter(best_epoch + 1, val_acc_best, color="red", marker="x", s=60, label="best")
    axs[1].set_title(f"{title} — Accuracy")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Accuracy")
    axs[1].legend()
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    print(f"Best epoch: {best_epoch + 1}")
    print(f"Val loss:   {min_val_loss:.4f}")
    print(f"Val acc:    {val_acc_best:.4f}")
    return best_epoch + 1, val_acc_best

best_ep, val_acc = plot_learning_curves(hist_baseline, "Baseline (Emb + AvgPool)")

In [ ]:
y_pred_base = evaluate_model(baseline, "Baseline (Emb+AvgPool)", X_test_vec, y_test, hist_baseline, train_time)
print()
print(classification_report(y_test, y_pred_base, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred_base)

confused = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm[i, j] > 0:
            confused.append((cm[i, j], class_names[i], class_names[j]))
confused.sort(reverse=True)

print("Top 10 confused pairs (true -> predicted, count):")
for count, true, pred in confused[:10]:
    print(f"  {true:25s} -> {pred:25s}  ({count})")

### Graded Questions (5 pts each)

1. **Model Architecture:**
   Describe your baseline model and justify why this structure suits your dataset.

2.1. The baseline is a `Sequential` model: `Embedding(20,000 × 64)` → `GlobalAveragePooling1D` → `Dense(64, relu)` → `Dense(31, softmax)`, totaling 1,286,175 parameters. This architecture was chosen because it matches the assignment's recommended text baseline (Embedding → GlobalAveragePooling → Dense → Softmax). GlobalAveragePooling1D averages all token embeddings into a single 64-dim vector, discarding word order — intentionally simple to establish a performance floor. The model takes pre-vectorized integer sequences as input (shape: batch × 64).

2. **Training Behavior:**
   Summarize the model’s training and validation curves. What trends did you observe?

2.2. Training ran for 16 epochs before early stopping (patience=5, triggered after best epoch 11). Training accuracy rose steadily from 0.137 (epoch 1) to 0.733 (epoch 16), while validation accuracy plateaued around epoch 10–11 at ~0.561 and stopped improving. Validation loss reached its minimum of 1.6550 at epoch 11 and began increasing afterward (1.6688 at epoch 13, 1.7228 at epoch 16), indicating overfitting. The widening gap between training accuracy (0.733) and validation accuracy (0.566) at the final epoch confirms the model memorized training patterns beyond what generalized. Total training time was 3 minutes 36 seconds on Colab CPU.

  3. **Baseline Metrics:**
   Report validation and test metrics. What does this performance tell you about dataset difficulty?

2.3. At the best epoch (11), validation accuracy was 0.5613 with a validation loss of 1.6550. On the held-out test set: accuracy = 0.5632, macro-F1 = 0.5000. The gap between weighted-F1 (0.58) and macro-F1 (0.50) reflects the class imbalance — the model performs better on large classes like POLITICS (F1=0.67) and STYLE (F1=0.77) but struggles with small or ambiguous ones like GOOD NEWS (F1=0.18), WEIRD NEWS (F1=0.26), and FIFTY (F1=0.26). The top confusion pair is HOME & LIVING → WELLNESS (210 misclassifications), followed by ENTERTAINMENT → COMEDY (195) and POLITICS → WORLD NEWS (191), which are semantically close categories. Overall, ~56% accuracy on 31 classes (vs. 3.2% random baseline) shows the model learned meaningful patterns, but the flat val accuracy curve suggests this architecture's capacity is nearly exhausted.

  4. **Reflection:**
   What are the main limitations of your baseline? Which specific improvements (depth, regularization, pretraining) would you try next?
  

2.4. The main limitation is that GlobalAveragePooling discards word order entirely — the model treats "dog bites man" and "man bites dog" identically. This ceiling is visible in the plateauing validation accuracy around 56%. The top confusions (HOME & LIVING ↔ WELLNESS, ENTERTAINMENT → COMEDY, POLITICS → WORLD NEWS) involve semantically overlapping categories where word order and context matter. Two concrete next steps: (1) add a Bidirectional LSTM or GRU layer before the dense head to capture sequential context, with dropout for regularization since the model is already overfitting; (2) in Problem 4, use a pretrained transformer (DistilBERT) that brings both contextual embeddings and prior language knowledge, which should help with the ambiguous category pairs.

## Problem 3 – Custom (Original) Model (20 pts)

### Goal

Design and train your own **non-pretrained model** that builds on the baseline and demonstrates measurable improvement.
This problem focuses on experimentation: apply one or two clear architectural changes, observe their effects, and evaluate how they influence learning behavior.


### Steps to Follow

1. **Modify or extend your baseline architecture**

   * Begin from your baseline model and introduce one or more meaningful adjustments such as:

     * Adding **dropout** or **batch normalization** for regularization.
     * Increasing **depth** (extra convolutional or dense layers).
     * Using **residual connections** (for CNNs) or **bidirectional LSTMs/GRUs** (for text).
     * Trying alternative activations like `ReLU`, `LeakyReLU`, or `GELU`.
   * Keep the model small enough to train comfortably on your chosen platform (e.g., Colab)

2. **Observe what specific limitations you want to address**

   * Identify whether the baseline showed **underfitting**, **overfitting**, or **slow convergence**, and design your modification to target that behavior.
   * Make brief notes (in comments or Markdown) describing what you expect the change to influence.

3. **Train and evaluate under the same conditions**

   * Use the **same data splits**, **random seed**, and **metrics** as in Problem 2.
   * Apply **early stopping** on validation loss.
   * Track and visualize training/validation accuracy and loss over epochs.

4. **Compare outcomes to the baseline**

   * Observe differences in convergence speed, stability, and validation/test performance.
   * Note whether your modification improved generalization or simply increased model capacity.

### Graded Questions (5 pts each)

1. **Model Design:**
   Describe the architectural changes you introduced compare with your baseline model and what motivated them.

3.1. **Your answer here:**



2. **Training Results:**
   Present key validation and test metrics. Did your modifications improve performance?

3.2. **Your answer here:**



3. **Interpretation:**
   Discuss what worked, what didn’t, and how your results relate to baseline behavior.

3.3. **Your answer here:**



4. **Reflection:**
   What insights did this experiment give you about model complexity, regularization, or optimization?

3.4. **Your answer here:**



## Problem 4 – Pretrained Model (Transfer Learning) (20 pts)

### Goal

Apply **transfer learning** to see how pretrained knowledge improves accuracy, convergence speed, and generalization.
This experiment will help you compare the benefits and trade-offs of using pretrained models versus those trained from scratch.


### Steps to Follow

1. **Select a pretrained architecture**

   * **Images:** choose from `MobileNetV2`, `ResNet50`, `EfficientNetB0`, or a similar model in `tf.keras.applications`.
   * **Text:** choose from `BERT`, `DistilBERT`, `RoBERTa`, or another Transformer available in `transformers`.

2. **Adapt the model for your dataset**

   * Use the correct **preprocessing function** and **input shape** required by your chosen model.
   * Replace the top layer with your own **classification head** (e.g., `Dense(num_classes, activation='softmax')`).

3. **Apply transfer learning**

   * Choose an appropriate **training strategy** for your pretrained model. Options include:

     * **Freezing** the pretrained base and training only a new classification head.
     * **Partially fine-tuning** selected upper layers of the base model.
     * **Full fine-tuning** (all layers trainable) with a reduced learning rate.
   * Adjust your learning rate schedule to match your strategy (e.g., smaller LR for fine-tuning).
   * Observe how your chosen approach affects **validation loss**, **training time**, and **model stability**.

4. **Train and evaluate under consistent conditions**

   * Use the same **splits**, **metrics**, and **evaluation protocol** as in earlier problems.
   * Record training duration, validation/test performance, and any resource constraints (GPU memory, runtime).

5. **Compare and analyze**

   * Observe how transfer learning changes both **performance** and **efficiency** relative to your baseline and custom models.
   * Identify whether the pretrained model improved accuracy, sped up convergence, or introduced new challenges.


**Note (Sergey):** If training feels slow on Apple Silicon, try setting `USE_GPU = True` in the setup cell and restarting the kernel. BERT-scale models benefit from Metal GPU, unlike the smaller P2-P3 models. TF locks the device config at startup, so a kernel restart is needed to switch.

### Graded Questions (5 pts each)

1. **Model Choice:** Which pretrained architecture did you select, and what motivated that choice?

In [ ]:
## import + set up
from transformers import AutoTokenizer
import tensorflow as tf

checkpoint = "distilbert-base-uncased"

MAX_LEN = 128
BATCH = 8 if COLAB_CPU else 16

In [ ]:
## tokenization - different from baseline line model tokenization
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_texts(texts, tokenizer, max_length=128):
    return tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="np"
    )

X_train_tok = tokenize_texts(X_train, tokenizer, MAX_LEN)
X_val_tok   = tokenize_texts(X_val, tokenizer, MAX_LEN)
X_test_tok  = tokenize_texts(X_test, tokenizer, MAX_LEN)

print("Train token shapes:")
for k, v in X_train_tok.items():
    print(f"{k}: {v.shape}")

train_ds = tf.data.Dataset.from_tensor_slices((
    {k: tf.convert_to_tensor(v) for k, v in X_train_tok.items()},
    y_train
)).shuffle(len(X_train), seed=SEED).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((
    {k: tf.convert_to_tensor(v) for k, v in X_val_tok.items()},
    y_val
)).batch(BATCH).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((
    {k: tf.convert_to_tensor(v) for k, v in X_test_tok.items()},
    y_test
)).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
from transformers import TFAutoModelForSequenceClassification, create_optimizer
import math
import tensorflow as tf

p4_model = TFAutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_classes   # IMPORTANT: multi-class (combined categories)
)

p4_model.trainable = True  # full fine-tuning

EPOCHS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

steps_per_epoch = math.ceil(len(X_train) / BATCH)
num_train_steps = steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_train_steps)

optimizer, lr_schedule = create_optimizer(
    init_lr=LR,
    num_warmup_steps=num_warmup_steps,
    num_train_steps=num_train_steps,
    weight_decay_rate=WEIGHT_DECAY
)

p4_model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

p4_model.summary()

In [ ]:
## same as baseline
# bug fixed here : 
import tf_keras
es = tf_keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

t0 = time.time()

hist_hf = p4_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[es],
    verbose=1
)

train_time = time.time() - t0
print(f"Training time: {train_time/60:.2f} minutes")

In [ ]:
## eval
from sklearn.metrics import classification_report, f1_score
import numpy as np

pred_output = p4_model.predict(test_ds)
logits = pred_output.logits

y_pred = np.argmax(logits, axis=-1)

test_acc = float(np.mean(y_pred == y_test))
test_f1 = float(f1_score(y_test, y_pred, average="macro"))

print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F1 (macro): {test_f1:.4f}")

print()
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

4.1. **Your answer here:**

The pretraining architecture we selected was `DistilRoBERTa`. This architecture provided the best balance of a quicker run time while not sacrificing accuracy. In comparison, `DistilBERT` would be used if run time was the only consideration and `deBERTa-v3` would be used if pure accuracy was all that mattered.


2. **Fine-Tuning Plan:** Describe your fine-tuning strategy and why you chose it.

4.2. **Your answer here:**

The fine tuning strategy we chose was full fine-tubing with a reduced learning rate. The HuffPost dataset is large enough to pursue this training option and through updating the entire encoder the task adaption tends to be better as well. When compared to only training a new head or by freezing a majority of it.



3. **Performance:** Report key metrics and compare them with your baseline and custom models.

4.3. **Your answer here:**



4. **Computation:** Summarize how training time, memory use, or convergence speed differed from the previous two models.

4.4. **Your answer here:**



## Problem 5 – Comparative Evaluation and Discussion (20 pts)

### Goal

Compare your **baseline**, **custom**, and **pretrained** models to evaluate how design choices affected performance, efficiency, and generalization.
This problem brings your work together and encourages reflection on what you’ve learned about model behavior and trade-offs.

**Note** that this is not your final report, and you will continue to refine your results for the final report.

### Steps to Follow

1. **Compile key results**

   * Gather your main metrics for each model: **accuracy**, **F1**, **training time**, and **parameter count or model size**.
   * Ensure all numbers come from the same evaluation protocol and test set.

2. **Visualize the comparison**

   * Present results in a **single, well-organized chart or table**.
   * Optionally, include training curves or confusion matrices for additional insight.

3. **Analyze comparative performance**

   * Observe which model performed best by your chosen metric(s).
   * Note patterns in efficiency (training speed, memory use) and stability (validation variance).

4. **Inspect model behavior**

   * Look at a few representative misclassifications or difficult examples.
   * Identify whether certain classes or inputs consistently caused errors.

5. **Plan forward improvements**

   * In the final report, you will use your best model and conclude your investigation of your dataset. Based on your observations, decide on a model and next steps for refining your approach in the final project (e.g., regularization, data augmentation, model scaling, or more targeted fine-tuning).

### Graded Questions (4 pts each)

1. **Summary Table and Performance Analysis:** Present a clear quantitative comparison of all three models. Which model achieved the best overall results, and what factors contributed to its success?

5.1. **Your answer here:**



2. **Trade-Offs:** Discuss how complexity, accuracy, and efficiency balanced across your models.

5.2. **Your answer here:**



3. **Error Patterns:** Describe the types of examples or classes that remained challenging for all models.

5.3. **Your answer here:**



4. **Next Steps:** Based on these findings, decide on a model to go forward with and outline your plan for improving that model.


5.4 **Your answer here:**



### Final Question: Describe what use you made of generative AI tools in preparing this Milestone.

**AI tools used:** Claude Code (Anthropic, Opus model) via CLI.

**How:** Technical support — environment setup (resolving TensorFlow/tensorflow-metal version conflicts, configuring GPU vs CPU execution for Apple Silicon), debugging runtime errors (PyArrow array indexing incompatibility with sklearn), and notebook infrastructure (results persistence across kernel restarts, tf.data pipeline setup).

**Why:** Platform-specific issues (M4 Pro / Metal GPU) are not covered in course materials and required troubleshooting outside the scope of the assignments.

**Full conversation log:** Available upon request.